# Sparse pairwise regression with MGS-KAN

Does a standalone MGS-KAN predict sparse pairwise functions more accurately than a two-hidden-layer MLP at a matched trainable parameter budget?

Run all cells from the thesis directory with a Python kernel that has PyTorch, NumPy and Matplotlib. The default output is the completed study. Set `output` to an unused directory for a fresh run. An interrupted run can resume in its original directory with the same code, configuration and environment.

## Frozen protocol

The MGS-KAN maps 12 features to a scalar by adding an affine term, four Gaussian kernels per feature, and two Gaussian kernels per candidate pair. Each pair kernel has a learned center, amplitude, and separate positive-definite precision $P=LL^\top$ with positive Cholesky diagonal. Training adds $\lambda\sum_{i<j}\|a_{ij}\|_2$ to mean squared error. The width-24 MLP has two ReLU hidden layers and a linear output.

Both targets use independent standard-normal inputs, unit population signal variance, and independent Gaussian noise with standard deviation 0.10. For a given seed, the two targets share the feature and noise draws. The sparse target combines a product, a sine of a product, a rotated Gaussian, and a smaller additive component on the remaining features. The additive control sums single-feature sine effects. The study's Gaussian is a labeled substitute fixture because the proposal's numeric constants were unavailable.

The models share data splits, train-only input standardization, minibatch order for each initialization seed, and the earliest checkpoint with minimum noisy validation RMSE. Each optimizer runs at most 150 epochs with patience 35. For each model, six tuning configurations are evaluated on both datasets with a dedicated tuning seed. The mean validation RMSE across the two datasets selects one configuration per model before the final fits.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from experiments.mgs_kan import config as c
from experiments.mgs_kan.experiment import DEFAULT_OUTPUT, run_study

output = DEFAULT_OUTPUT
print(f"Split sizes (train, validation, test) = {c.SIZES}")
print(f"Data seeds = {c.DATA_SEEDS}. Initialization seeds = {c.INIT_SEEDS}.")
print(f"Planted pairs (0-based) = {c.PLANTED}. Selection threshold = {c.SELECTION_FRACTION:.0%} of training target SD.")

## Checks, pilot, tuning and final fits

The float64 gate tests pair indexing, positive definiteness, gradients, kernel orientation and marginal subtraction. The pilot tests whether both models can fit the same small noiseless sample for each target. Its reported error is training RMSE. Each final fit saves the chosen parameters, held-out predictions and elapsed training time.

In [ ]:
paired = run_study(output)


def read_json(path):
    return json.loads(path.read_text())


checks = read_json(output / "checks.json")
pilot = read_json(output / "pilot.json")
tuning = read_json(output / "tuning.json")
print(f"Trainable parameters = {checks['parameter_counts']}")
print(f"Rotated-Gaussian maximum absolute error = {checks['rotated_gaussian_max_abs_error']:.3g}")
for row in pilot:
    print(f"Pilot {row['dataset']:8} {row['model']:7} training RMSE = {row['training_rmse']:.4g}")
print(f"Validation-selected settings = {tuning['selected']}")

## Held-out prediction

Each row first averages the three test RMSEs for each model within a data seed, then subtracts MLP from MGS-KAN. A negative difference favors MGS-KAN. The three rows per dataset show the independent paired comparisons.

In [ ]:
def show_table(headers, rows):
    lines = [headers, ["---"] * len(headers), *rows]
    display(Markdown("\n".join("| " + " | ".join(map(str, line)) + " |" for line in lines)))


show_table(
    ["Dataset", "Data seed", "MGS-KAN RMSE", "MLP RMSE", "MGS-KAN − MLP"],
    [(r["dataset"], r["data_seed"], f"{r['mgs_kan_rmse']:.4f}", f"{r['mlp_rmse']:.4f}",
      f"{r['mgs_minus_mlp']:+.4f}") for r in paired],
)
fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharex=True, layout="constrained")
for ax, dataset in zip(axes, c.DATASETS):
    group = [r for r in paired if r["dataset"] == dataset]
    ax.axvline(0, color="0.65", linestyle="--")
    ax.scatter([r["mgs_minus_mlp"] for r in group], range(len(group)), color="#1d6475", s=45, zorder=3)
    ax.set_yticks(range(len(group)), [str(r["data_seed"]) for r in group])
    ax.set(title=dataset.capitalize(), xlabel="MGS-KAN − MLP test RMSE")
axes[0].set_ylabel("Data seed")
plt.show()

## Pair recovery and false selections

For each learned pair function $g_{ij}$, the pure interaction is $g_{ij}(x_i,x_j)-E[g_{ij}\mid X_i=x_i]-E[g_{ij}\mid X_j=x_j]+E[g_{ij}]$. The code computes the expectations analytically under the independent Gaussian population in training-scaled coordinates. The score is the component's RMS on validation rows. The cutoff is 5% of the noisy training-target standard deviation. Top-three recovery depends only on rank. On the additive control, every selected pair is a false selection.

In [ ]:
fits = sorted((output / "fits").glob("*/metrics.json"))
show_table(
    ["Dataset", "Data seed", "Init", *(f"Rank {pair}" for pair in c.PLANTED), "Top 3 hits", "Selected", "False selected"],
    [
        (info["dataset"], info["data_seed"], info["init_seed"],
         *(ranks.get(pair, "—") for pair in c.PLANTED), info["top_three_hits"] if ranks else "—",
         info["selected_pairs"], info["false_selections"])
        for path in fits if (info := read_json(path))["model"] == "mgs_kan"
        for ranks in [{tuple(p["pair"]): p["rank"] for p in read_json(path.parent / "pairs.json")["pairs"] if p["planted"]}]
    ],
)

## Training cost

The table summarizes wall-clock seconds for the final fits on the same CPU and thread count. Timing includes optimization, validation and loading the selected checkpoint. Per-fit times are in `runs.csv`.

In [ ]:
elapsed = {model: [] for model in c.MODELS}
for path in fits:
    info = read_json(path)
    elapsed[info["model"]].append(info["training_seconds"])
show_table(["Model", "Fits", "Median seconds", "Total seconds"],
           [(model, len(times), f"{np.median(times):.2f}", f"{sum(times):.2f}") for model, times in elapsed.items()])

## Verdict

In [ ]:
verdict = output / "verdict.md"
if verdict.exists():
    display(Markdown(verdict.read_text().split("\n", 1)[1]))
else:
    print("See the paired RMSE, pair recovery and training cost above for this run.")